# DriftWise: gate trade-off sweep
How much accuracy does each gate setting give up under drift, and how many harmful updates does it avoid when nothing has changed?
Compares the sequential gate (5 significance levels) with a simple point-estimate rule (6 margins).

**How to run:**
1. Run cell 1 (install). Then **Runtime > Restart session**.
2. Do NOT run cell 1 again. Run cell 2 (allow Google Drive access when asked).
3. Wait about 10 to 15 minutes and paste the printed table back into the chat (and describe or screenshot the plot).

Needs `ega_pipeline.py` and `ega_gate.py` in `MyDrive/DriftWise/src/`.

## 1. Install (run once, then Runtime > Restart session)

In [ ]:
!pip install river "numpy<2.3" -q

## 2. Sweep (Part A: pattern drift, Part B: no-drift streams)

In [ ]:
import sys, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/DriftWise/src")
import ega_pipeline as ep                       # all our functions, restored by one import
seq_gate = ep.sequential_gate                   # the real sequential gate (kept so we can restore it)

def bal(y, p):
    r = [float((p[y == c] == c).mean()) for c in (0, 1)]
    return float(np.mean(r)), float(min(r))

def make_seq(A):
    """Sequential gate with total significance level A (0.05 = the setting we have used so far)."""
    def gate(y, p_new, p_old, look=None, alpha_total=0.05, **k):
        return seq_gate(y, p_new, p_old, look=look, alpha_total=alpha_total * (A / 0.05))
    return gate

def make_point(margin, min_n=200, min_per_class=30):
    """Simple switch: accept if the candidate's balanced accuracy beats the current model's by more than `margin`."""
    def gate(y, p_new, p_old, look=None, alpha_total=None, **k):
        y, p_new, p_old = map(np.asarray, (y, p_new, p_old))
        n1 = int(y.sum()); n0 = len(y) - n1
        if len(y) < min_n or n1 < min_per_class or n0 < min_per_class:
            return {"decision": "DEFER", "est_balacc": 0.0}
        diff = bal(y, p_new)[0] - bal(y, p_old)[0]
        return {"decision": "ACCEPT" if diff > margin else "REJECT", "est_balacc": diff}
    return gate

GATES = {}
for A in [0.01, 0.05, 0.2, 0.5, 0.9]:
    GATES[f"sequential, alpha={A}"] = make_seq(A)
for m in [0, 0.01, 0.02, 0.03, 0.05, 0.08]:
    GATES[f"point, margin={m}"] = make_point(m)

# ---------------- Part A: pattern drift, 20 fresh seeds ----------------
def run_with_gate(gate, X, y):
    ep.sequential_gate = gate                    # run_ega_v2 looks this name up when it runs
    try:
        ep.reset_cost()
        r = ep.run_ega_v2(X, y, lag=0, patience=3)
        r["models_trained"] = ep.COST["fits"]
    finally:
        ep.sequential_gate = seq_gate
    return r

SEEDS_A = range(10, 30)
rows = []
for i, seed in enumerate(SEEDS_A, start=1):
    X, y = ep.make_stream(seed, "pattern_drift")
    for name, g in GATES.items():
        rows.append({"gate": name, "seed": seed, **run_with_gate(g, X, y)})
    print(f"Part A: seed {i}/{len(SEEDS_A)} done")
A = pd.DataFrame(rows)

# ---------------- Part B: streams with NO drift; do the gates accept truly worse models? ----------------
def make_no_drift_stream(seed, n=6000):
    r = np.random.default_rng(seed)
    y = (r.random(n) < 0.05).astype(int)
    X = r.normal(size=(n, 2))
    X[:, 0] += 2.0 * y
    return X, y

val_size, gap, train_size = 1000, 50, 1000
checkpoints = list(range(4500, 6000, 200))
n_streams = 200
rec = []
for seed in range(n_streams):
    X, y = make_no_drift_stream(seed)
    Xt, yt = make_no_drift_stream(seed + 100000, n=20000)       # big fresh test set = the 'truth'
    old = ep.fit_model(X[:3000], y[:3000])
    b_old = bal(yt, ep.predict(old, Xt))[0]
    for look, t in enumerate(checkpoints, start=1):
        te = t + 1 - val_size - gap
        cand = ep.fit_model(X[te - train_size:te], y[te - train_size:te])
        Xv, yv = X[t + 1 - val_size:t + 1], y[t + 1 - val_size:t + 1]
        p_old, p_new = ep.predict(old, Xv), ep.predict(cand, Xv)
        true_diff = bal(yt, ep.predict(cand, Xt))[0] - b_old     # real change in balanced accuracy
        for name, g in GATES.items():
            acc = g(yv, p_new, p_old, look=look)["decision"] == "ACCEPT"
            rec.append({"gate": name, "seed": seed, "accept": acc, "true_diff": true_diff})
B = pd.DataFrame(rec)
n_checks = n_streams * len(checkpoints)

b_rows = []
for name in GATES:
    s = B[B.gate == name]; a = s[s.accept]
    harmful = a[a.true_diff < -0.01]
    b_rows.append({"gate": name,
                   "accepts /1000 checks": round(1000 * len(a) / n_checks, 1),
                   "harmful accepts /1000 checks": round(1000 * len(harmful) / n_checks, 1),
                   "% streams with harmful accept": round(100 * harmful.seed.nunique() / n_streams, 1)})
B_sum = pd.DataFrame(b_rows).set_index("gate")
A_sum = A.groupby("gate")[["balanced_acc", "worst_class_recall", "n_updates", "models_trained"]].mean().round(3)
summary = A_sum.join(B_sum).loc[list(GATES)]
pd.set_option("display.width", 300, "display.max_columns", 20)
print("\nTRADE-OFF TABLE (A = under drift, 20 seeds; B = no drift, 200 streams x 8 checks)")
display(summary)
print("Consistency check against the last ablation: 'sequential, alpha=0.05' should be near 31 accepts and 1.2 harmful per 1000;"
      " 'point, margin=0' near 336 and 58.8.")

# ---------------- Plot: the trade-off curve ----------------
fig, ax = plt.subplots(figsize=(8, 5.5))
for prefix, color in [("sequential", "tab:blue"), ("point", "tab:orange")]:
    d = summary[[i.startswith(prefix) for i in summary.index]]
    ax.plot(d["harmful accepts /1000 checks"], d["balanced_acc"], "o-", color=color, label=prefix + " gate" if prefix == "sequential" else "simple point rule")
    for label, r in d.iterrows():
        ax.annotate(label.split(", ")[1], (r["harmful accepts /1000 checks"], r["balanced_acc"]),
                    textcoords="offset points", xytext=(5, 4), fontsize=8)
ax.set_xscale("symlog", linthresh=1)
ax.set_xlabel("Harmful accepts per 1000 checks (no-drift streams; lower = safer)")
ax.set_ylabel("Balanced accuracy under drift (higher = better)")
ax.set_title("Safety vs accuracy: the gate settings compared with simple switch rules")
ax.legend()
plt.tight_layout()
os.makedirs("/content/drive/MyDrive/DriftWise/results", exist_ok=True)
fig.savefig("/content/drive/MyDrive/DriftWise/results/gate_tradeoff_curve.png", dpi=150)
plt.show()

summary.to_csv("/content/drive/MyDrive/DriftWise/results/gate_tradeoff_summary.csv")
A.to_csv("/content/drive/MyDrive/DriftWise/results/gate_sweep_pattern_drift.csv", index=False)
B.to_csv("/content/drive/MyDrive/DriftWise/results/gate_sweep_no_drift.csv", index=False)
print("Saved: gate_tradeoff_summary.csv, gate_tradeoff_curve.png and the two raw CSVs")